# LeetCode 799 — Champagne Tower bằng Python

Link đề: https://leetcode.com/problems/champagne-tower/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def champagneTower(self, poured: int, query_row: int, query_glass: int) -> float:
        ...
```

Ghi chú: kết quả là **số thực**. LeetCode chấp nhận sai số nhỏ (khoảng `1e-5`); trong notebook ta so sánh với một lời giải **chính xác bằng số nguyên/`Fraction`** (không sai số làm tròn) và dùng dung sai `1e-9`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một tháp ly hình tam giác; rót `poured` cốc champagne vào ly trên cùng; phần **tràn** chảy **đều** xuống hai ly dưới bên trái/phải; tính ly `(query_row, query_glass)` đầy bao nhiêu (tối đa `1`).
2. Nắm các công cụ Python: chỉ số `(i, j)` của tháp tam giác (ly `(i, j)` đổ xuống `(i+1, j)` và `(i+1, j+1)`), `min(1, x)`, ma trận `[[0] * 101 for _ in range(101)]`, phép chia thực `/` và số thực dấu phẩy động, `fractions.Fraction` để tính **chính xác**.
3. Đi từ **mô phỏng từng cốc** → **đệ quy/memo theo ly** → **DP hai chiều theo hàng** → **DP một chiều (cuộn hàng)** → **DP số nguyên đã nhân tỉ lệ `2^i` (chính xác)**.
4. Hiểu mấu chốt: **không cần mô phỏng thứ tự rót**; chỉ cần biết **tổng lượng chảy vào mỗi ly**: ly có tổng lượng `x` giữ `min(1, x)` và đẩy `max(0, x - 1) / 2` xuống **mỗi** ly con.
5. Tránh các lỗi kinh điển: quên **trừ 1** (ly đầy giữ 1 cốc) trước khi chia đôi, quên giới hạn kết quả ở `1`, mảng quá nhỏ gây `IndexError`, dùng `//` thay `/`, và cộng lượng **âm** xuống các ly chưa đầy.

## 2. Đề bài & ví dụ

Ta xếp các ly thành một tháp hình tam giác: hàng thứ nhất có `1` ly, hàng thứ hai có `2` ly, ..., tới hàng thứ `100`. Mỗi ly chứa được **đúng một cốc** champagne.

Rót champagne vào ly trên cùng. Khi ly trên cùng đầy, phần **tràn** chảy **đều** xuống hai ly ngay bên dưới (trái và phải). Khi các ly đó đầy, phần tràn tiếp tục chảy đều xuống hai ly bên dưới chúng, và cứ thế. Ly ở hàng cuối cùng thì phần tràn rơi xuống sàn.

Ví dụ: rót `1` cốc → ly trên cùng đầy. Rót `2` cốc → hai ly hàng thứ hai mỗi ly có nửa cốc. Rót `3` cốc → hai ly đó đầy (tổng `3` ly đầy). Rót `4` cốc → hàng thứ ba: ly giữa đầy một nửa, hai ly ngoài mỗi ly một phần tư.

Cho `poured` (số cốc nguyên không âm), hãy trả về ly thứ `query_glass` ở hàng thứ `query_row` (cả hai đánh số từ `0`) đầy bao nhiêu.

**Ví dụ 1**

- Input: `poured = 1`, `query_row = 1`, `query_glass = 1`
- Output: `0.00000`
- Giải thích: một cốc rót vào ly trên cùng, không có phần tràn nên các ly dưới trống.

**Ví dụ 2**

- Input: `poured = 2`, `query_row = 1`, `query_glass = 1`
- Output: `0.50000`
- Giải thích: ly trên cùng dư một cốc; hai ly hàng dưới chia đôi, mỗi ly nhận nửa cốc.

**Ví dụ 3**

- Input: `poured = 100000009`, `query_row = 33`, `query_glass = 17`
- Output: `1.00000`

**Ràng buộc:**

- `0 <= poured <= 10^9`
- `0 <= query_glass <= query_row < 100`

Với `poured` tới `10^9` ta **không thể** mô phỏng từng cốc một cách ngây thơ.

In [1]:
from typing import List
from fractions import Fraction
from functools import cache
import random

examples = [
    ((1, 1, 1), 0.0),
    ((2, 1, 1), 0.5),
    ((100000009, 33, 17), 1.0),
]
for args, expected in examples:
    print("poured, query_row, query_glass =", args, "-> kỳ vọng", expected)

poured, query_row, query_glass = (1, 1, 1) -> kỳ vọng 0.0
poured, query_row, query_glass = (2, 1, 1) -> kỳ vọng 0.5
poured, query_row, query_glass = (100000009, 33, 17) -> kỳ vọng 1.0


## 3. Kiến thức nền

### 3.1. Tháp tam giác và chỉ số `(i, j)`

Hàng `i` (từ `0`) có `i + 1` ly, đánh số cột `j = 0..i`. Ly `(i, j)` đổ phần tràn xuống **hai** ly: `(i + 1, j)` (con trái) và `(i + 1, j + 1)` (con phải). Ngược lại, ly `(i, j)` nhận từ **hai** cha `(i - 1, j - 1)` và `(i - 1, j)` (nếu tồn tại: ly ngoài rìa chỉ có một cha).

In [2]:
rows = 4
for i in range(rows):
    print(" " * (rows - i) * 2 + "   ".join(f"({i},{j})" for j in range(i + 1)))
print()
i, j = 2, 1
print(f"ly ({i},{j}) đổ xuống: ({i + 1},{j}) và ({i + 1},{j + 1})")
print(f"ly ({i},{j}) nhận từ: ({i - 1},{j - 1}) và ({i - 1},{j})")

        (0,0)
      (1,0)   (1,1)
    (2,0)   (2,1)   (2,2)
  (3,0)   (3,1)   (3,2)   (3,3)

ly (2,1) đổ xuống: (3,1) và (3,2)
ly (2,1) nhận từ: (1,0) và (1,1)


### 3.2. Phần tràn và công thức chia đôi

Giả sử tổng lượng chảy **vào** một ly là `x` (cốc). Ly chứa được tối đa `1`:

- Nếu `x <= 1`: ly giữ `x`, **không có tràn**.
- Nếu `x > 1`: ly giữ `1` và **tràn** `x - 1`; phần tràn chia **đều**, mỗi ly con nhận `(x - 1) / 2`.

Gộp: lượng truyền xuống mỗi con là `max(0, x - 1) / 2`; lượng ly giữ là `min(1, x)`.

In [3]:
def keep_and_pass(x):
    return min(1, x), max(0, x - 1) / 2

for x in [0, 0.5, 1, 2, 3.5]:
    keep, down = keep_and_pass(x)
    print(f"vào {x}: giữ {keep}, mỗi ly con nhận {down}")

vào 0: giữ 0, mỗi ly con nhận 0.0
vào 0.5: giữ 0.5, mỗi ly con nhận 0.0
vào 1: giữ 1, mỗi ly con nhận 0.0
vào 2: giữ 1, mỗi ly con nhận 0.5
vào 3.5: giữ 1, mỗi ly con nhận 1.25


### 3.3. Số thực dấu phẩy động và `Fraction`

Số thực `float` lưu gần đúng theo nhị phân: `0.1 + 0.2` không đúng bằng `0.3`. Tuy nhiên các phân số có mẫu là **lũy thừa của 2** (`1/2`, `1/4`, `3/8`, ...) biểu diễn **chính xác** trong `float` (miễn là tử số không quá lớn). Bài này chỉ chia đôi nên rất hợp. Để kiểm chứng ta dùng `fractions.Fraction`: số hữu tỉ **chính xác**, không có sai số làm tròn.

So sánh số thực nên dùng **dung sai**: `abs(a - b) < 1e-9`, không dùng `==`.

In [4]:
print(0.1 + 0.2 == 0.3, 0.1 + 0.2)
print(1 / 2 + 1 / 4 == 0.75)
f = Fraction(1, 3) + Fraction(1, 6)
print(f, float(f), f == Fraction(1, 2))
print(abs((0.1 + 0.2) - 0.3) < 1e-9)
print(Fraction(7, 8) < 1, min(Fraction(3, 2), 1))

False 0.30000000000000004
True
1/2 0.5 True
True
True 1


### 3.4. Ma trận `101 x 101` và vì sao cần dư một hàng/cột

Nếu cài DP theo ma trận `f[i][j]`, khi xử lý hàng `i` ta ghi vào `f[i + 1][j]` và `f[i + 1][j + 1]`. Với `i` lên tới `99` ta ghi vào `f[100][...]` và cột `100` → mảng phải kích thước **`101 x 101`**, không phải `100 x 100`. Tạo ma trận độc lập bằng `[[0] * 101 for _ in range(101)]` (không dùng `[[0] * 101] * 101` vì các hàng sẽ là cùng một list).

In [5]:
f = [[0] * 101 for _ in range(101)]
f[100][100] = 1
print(len(f), len(f[0]), f[100][100])

bad = [[0] * 3] * 3          # SAI: ba hàng là cùng một đối tượng
bad[0][1] = 7
print(bad)

101 101 1
[[0, 7, 0], [0, 7, 0], [0, 7, 0]]


### 3.5. Chia `/` và `//`

`/` là chia thực (`5 / 2 = 2.5`), `//` là chia nguyên (`5 // 2 = 2`). Bài này cần **chia thực**: `(x - 1) / 2`. Dùng `//` sẽ làm mất các phần nửa/phần tư.

In [6]:
print(5 / 2, 5 // 2, (3 - 1) / 2, (2 - 1) // 2)

2.5 2 1.0 0


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `poured = 4`:

- **Hàng 0**: ly `(0,0)` nhận `4`. Giữ `1`, tràn `3`, mỗi con nhận `1.5`.
- **Hàng 1**: mỗi ly nhận `1.5`. Giữ `1`, tràn `0.5`, mỗi con nhận `0.25`.
- **Hàng 2**: ly `(2,0)` nhận `0.25` (chỉ từ `(1,0)`). Ly `(2,1)` nhận `0.25 + 0.25 = 0.5` (từ **hai** cha). Ly `(2,2)` nhận `0.25`.

```
hàng 0:           1
hàng 1:        1     1
hàng 2:     0.25  0.5  0.25
```

Khớp với đề: "ly giữa đầy một nửa, hai ly ngoài mỗi ly một phần tư".

**Quan sát:** lượng vào mỗi ly chỉ phụ thuộc **tổng tràn của hai cha** ở hàng trên. Hàng `i` chỉ phụ thuộc hàng `i - 1`. Đây là cấu trúc **quy hoạch động theo hàng**.

In [7]:
def pour_table(poured, rows):
    """In lượng CHẢY VÀO (trước khi cắt ở 1) và lượng ly giữ, theo từng hàng."""
    row = [float(poured)]
    for i in range(rows):
        print(f"hàng {i}: vào  =", [round(x, 4) for x in row])
        print(f"        giữ  =", [round(min(1, x), 4) for x in row])
        nxt = [0.0] * (len(row) + 1)
        for j, x in enumerate(row):
            if x > 1:
                nxt[j] += (x - 1) / 2
                nxt[j + 1] += (x - 1) / 2
        row = nxt

pour_table(4, 3)

hàng 0: vào  = [4.0]
        giữ  = [1]
hàng 1: vào  = [1.5, 1.5]
        giữ  = [1, 1]
hàng 2: vào  = [0.25, 0.5, 0.25]
        giữ  = [0.25, 0.5, 0.25]


## 5. Cách 1 — Brute Force: mô phỏng từng cốc

### Ý tưởng

Đúng như đề tả: rót **từng cốc một** vào ly trên cùng. Hàm `add(i, j, x)` đổ `x` cốc vào ly `(i, j)`: nếu còn chỗ (`x <=` chỗ trống) thì ly nhận hết; nếu không, ly đầy và phần dư chia đôi đổ xuống hai ly con (đệ quy). Rót đủ `poured` lần. Ta chỉ cần mô phỏng tới hàng `query_row` (phần tràn từ hàng này xuống dưới không ảnh hưởng tới nó).

### Code

In [8]:
def champagne_cups(poured, query_row, query_glass):
    rows = query_row + 1
    g = [[0.0] * (i + 1) for i in range(rows)]        # g[i][j] = lượng ly đang giữ (<= 1)

    def add(i, j, x):
        space = 1.0 - g[i][j]
        if x <= space:
            g[i][j] += x
            return
        g[i][j] = 1.0
        extra = x - space
        if i + 1 < rows:
            add(i + 1, j, extra / 2)
            add(i + 1, j + 1, extra / 2)

    for _ in range(poured):
        add(0, 0, 1.0)
    return g[query_row][query_glass]

for args, expected in examples[:2]:
    got = champagne_cups(*args)
    print(args, "->", got)
    assert abs(got - expected) < 1e-9
print(champagne_cups(4, 2, 1), champagne_cups(4, 2, 0), champagne_cups(25, 6, 1))

(1, 1, 1) -> 0.0
(2, 1, 1) -> 0.5
0.5 0.25 0.1875


### Độ phức tạp và nút thắt

- Mỗi cốc có thể kích hoạt đệ quy qua nhiều ly. Time ≈ `O(poured * rows^2)` tệ nhất. Space `O(rows^2)`.
- Với `poured = 10^9`: **hoàn toàn không khả thi**.

**Nút thắt:** ta mô phỏng **từng cốc**, nhưng kết quả cuối cùng không phụ thuộc cách chia nhỏ việc rót — chỉ phụ thuộc **tổng lượng** chảy vào mỗi ly. Hãy đo chi phí thật:

In [9]:
def count_add_calls(poured, query_row):
    rows = query_row + 1
    g = [[0.0] * (i + 1) for i in range(rows)]
    calls = 0

    def add(i, j, x):
        nonlocal calls
        calls += 1
        space = 1.0 - g[i][j]
        if x <= space:
            g[i][j] += x
            return
        g[i][j] = 1.0
        extra = x - space
        if i + 1 < rows:
            add(i + 1, j, extra / 2)
            add(i + 1, j + 1, extra / 2)

    for _ in range(poured):
        add(0, 0, 1.0)
    return calls

for p in [10, 100, 1000, 5000]:
    print(f"poured = {p:5d}, query_row = 9: số lần gọi add = {count_add_calls(p, 9)}")

poured =    10, query_row = 9: số lần gọi add = 108
poured =   100, query_row = 9: số lần gọi add = 67850


poured =  1000, query_row = 9: số lần gọi add = 985114


poured =  5000, query_row = 9: số lần gọi add = 5077114


## 6. Ý tưởng thuật toán / bước đột phá

**Gộp cả quá trình rót thành một "dòng chảy tổng".** Gọi `inflow(i, j)` là **tổng lượng** chảy vào ly `(i, j)` nếu ta rót `poured` cốc (không quan tâm thứ tự). Khi đó:

- `inflow(0, 0) = poured`.
- Ly `(i, j)` đẩy xuống **mỗi con** `max(0, inflow(i, j) - 1) / 2`.
- Do đó `inflow(i, j) = out(i-1, j-1) + out(i-1, j)` với `out(a, b) = max(0, inflow(a, b) - 1) / 2` (cha không tồn tại thì tính `0`).
- Đáp án: `min(1, inflow(query_row, query_glass))`.

Vì sao được phép bỏ qua thứ tự rót? Chất lỏng chỉ **cộng dồn** và các ly chỉ có hai trạng thái ("chưa đầy: giữ hết" / "đầy: tràn phần dư"). Tổng lượng đi vào mỗi ly sau khi rót xong không phụ thuộc cách chia nhỏ — ta sẽ **kiểm chứng bằng code** ở mục 10 (so với mô phỏng từng cốc).

Với công thức này, mỗi ly chỉ cần tính **một lần**: bài toán thành DP trên `rows * (rows + 1) / 2` ly, tối đa khoảng `5000` ly với `query_row < 100`.

In [10]:
# Cài đặt thẳng công thức: tính inflow bằng đệ quy (CHƯA nhớ, để thấy ý tưởng)
def champagne_formula(poured, query_row, query_glass):
    def inflow(i, j):
        if j < 0 or j > i:
            return 0
        if i == 0:
            return poured
        return max(0, (inflow(i - 1, j - 1) - 1) / 2) + max(0, (inflow(i - 1, j) - 1) / 2)
    return min(1, inflow(query_row, query_glass))

for args, expected in examples[:2]:
    got = champagne_formula(*args)
    print(args, "->", got)
    assert abs(got - expected) < 1e-9
for p in range(0, 30):
    for r in range(0, 5):
        for q in range(r + 1):
            assert abs(champagne_formula(p, r, q) - champagne_cups(p, r, q)) < 1e-12
print("công thức khớp mô phỏng từng cốc với poured < 30, row < 5")

(1, 1, 1) -> 0
(2, 1, 1) -> 0.5
công thức khớp mô phỏng từng cốc với poured < 30, row < 5


## 7. Các cách tiếp theo

Phần này có thêm 4 cách (tổng 5 cách): **đệ quy có nhớ**, **DP 2 chiều** (lời giải nộp), **DP cuộn hàng (một chiều)**, **DP số nguyên nhân `2^i` (chính xác)**.

### 7.1. Cách 2 — Đệ quy có nhớ (memo) theo ly

**Giải thích từng đoạn.**

1. `inflow(i, j)` như mục 6 nhưng dùng `@cache` để mỗi ly `(i, j)` chỉ tính **một lần**.
2. Không có memo thì đệ quy gọi lặp (mỗi ly được hỏi bởi nhiều con): số lần gọi tăng **mũ** theo hàng (mục dưới sẽ đếm).
3. Kết quả: `min(1, inflow(query_row, query_glass))`.

In [11]:
def champagne_memo(poured, query_row, query_glass):
    @cache
    def inflow(i, j):
        if j < 0 or j > i:
            return 0
        if i == 0:
            return poured
        return max(0, (inflow(i - 1, j - 1) - 1) / 2) + max(0, (inflow(i - 1, j) - 1) / 2)

    result = min(1, inflow(query_row, query_glass))
    inflow.cache_clear()
    return result

for args, expected in examples:
    got = champagne_memo(*args)
    print(args, "->", got)
    assert abs(got - expected) < 1e-9

(1, 1, 1) -> 0
(2, 1, 1) -> 0.5
(100000009, 33, 17) -> 1


In [12]:
# Đếm số lần gọi: không nhớ (mũ) so với có nhớ (đa thức)
def calls_without_memo(query_row):
    calls = 0
    def inflow(i, j):
        nonlocal calls
        calls += 1
        if j < 0 or j > i:
            return 0
        if i == 0:
            return 10**9
        return max(0, (inflow(i - 1, j - 1) - 1) / 2) + max(0, (inflow(i - 1, j) - 1) / 2)
    inflow(query_row, query_row // 2)
    return calls

def calls_with_memo(query_row):
    seen = set()
    def inflow(i, j):
        if (i, j) in seen:
            return
        seen.add((i, j))
        if j < 0 or j > i or i == 0:
            return
        inflow(i - 1, j - 1); inflow(i - 1, j)
    inflow(query_row, query_row // 2)
    return len(seen)

for r in [5, 10, 15, 20]:
    print(f"query_row = {r:2d}: không memo = {calls_without_memo(r):8d} lần gọi, có memo = {calls_with_memo(r)} ly khác nhau")

query_row =  5: không memo =       49 lần gọi, có memo = 17 ly khác nhau
query_row = 10: không memo =     1343 lần gọi, có memo = 46 ly khác nhau
query_row = 15: không memo =    35749 lần gọi, có memo = 87 ly khác nhau
query_row = 20: không memo =  1041351 lần gọi, có memo = 141 ly khác nhau


In [13]:
class Solution:
    def champagneTower(self, poured: int, query_row: int, query_glass: int) -> float:
        from functools import cache

        @cache
        def inflow(i, j):
            if j < 0 or j > i:
                return 0
            if i == 0:
                return poured
            return max(0, (inflow(i - 1, j - 1) - 1) / 2) + max(0, (inflow(i - 1, j) - 1) / 2)

        return min(1, inflow(query_row, query_glass))

print(Solution().champagneTower(2, 1, 1))

0.5


**Độ phức tạp:** Time `O(query_row^2)` (mỗi ly một lần), Space `O(query_row^2)` cho bảng nhớ và `O(query_row)` ngăn xếp đệ quy (tối đa `99`, an toàn).

### 7.2. Cách 3 — DP hai chiều theo hàng (lời giải nộp)

**Giải thích từng đoạn.**

1. `f = [[0] * 101 for _ in range(101)]`; `f[0][0] = poured`. Ở đây `f[i][j]` là **lượng chảy vào** ly `(i, j)`.
2. Duyệt hàng `i` từ `0` tới `query_row`, cột `j` từ `0` tới `i`. Nếu `f[i][j] > 1`: `half = (f[i][j] - 1) / 2`; đặt `f[i][j] = 1` (ly giữ 1); cộng `half` vào `f[i+1][j]` và `f[i+1][j+1]`.
3. Trả về `f[query_row][query_glass]`. Vì ly ở hàng `query_row` cũng được xử lý (vòng lặp gồm `i = query_row`) nên giá trị đã bị **cắt về tối đa `1`**.

In [14]:
def champagne_dp2d(poured, query_row, query_glass):
    f = [[0] * 101 for _ in range(101)]
    f[0][0] = poured
    for i in range(query_row + 1):
        for j in range(i + 1):
            if f[i][j] > 1:
                half = (f[i][j] - 1) / 2
                f[i][j] = 1
                f[i + 1][j] += half
                f[i + 1][j + 1] += half
    return f[query_row][query_glass]

for args, expected in examples:
    got = champagne_dp2d(*args)
    print(args, "->", got)
    assert abs(got - expected) < 1e-9

(1, 1, 1) -> 0
(2, 1, 1) -> 0.5
(100000009, 33, 17) -> 1


In [15]:
class Solution:
    def champagneTower(self, poured: int, query_row: int, query_glass: int) -> float:
        f = [[0] * 101 for _ in range(101)]
        f[0][0] = poured
        for i in range(query_row + 1):
            for j in range(i + 1):
                if f[i][j] > 1:
                    half = (f[i][j] - 1) / 2
                    f[i][j] = 1
                    f[i + 1][j] += half
                    f[i + 1][j + 1] += half
        return f[query_row][query_glass]

print(Solution().champagneTower(2, 1, 1), Solution().champagneTower(100000009, 33, 17))

0.5 1


**Độ phức tạp:** Time `O(query_row^2)` (tối đa khoảng `5000` ly), Space `O(1)` thêm cho kích thước cố định `101 x 101` (hay `O(query_row^2)` nếu cấp phát vừa đủ).

### 7.3. Cách 4 — DP một chiều (cuộn hàng)

**Ý tưởng.** Hàng `i + 1` chỉ phụ thuộc hàng `i`, nên chỉ cần giữ **một hàng** `row` (độ dài `i + 1`). Mỗi bước tạo `nxt` độ dài `i + 2` từ `row`. Space chỉ `O(query_row)`. Lặp `query_row` lần rồi cắt `min(1, row[query_glass])`.

In [16]:
def champagne_rolling(poured, query_row, query_glass):
    row = [poured]
    for _ in range(query_row):
        nxt = [0] * (len(row) + 1)
        for j, x in enumerate(row):
            if x > 1:
                half = (x - 1) / 2
                nxt[j] += half
                nxt[j + 1] += half
        row = nxt
    return min(1, row[query_glass])

for args, expected in examples:
    got = champagne_rolling(*args)
    print(args, "->", got)
    assert abs(got - expected) < 1e-9

(1, 1, 1) -> 0
(2, 1, 1) -> 0.5
(100000009, 33, 17) -> 1


In [17]:
class Solution:
    def champagneTower(self, poured: int, query_row: int, query_glass: int) -> float:
        row = [poured]
        for _ in range(query_row):
            nxt = [0] * (len(row) + 1)
            for j, x in enumerate(row):
                if x > 1:
                    half = (x - 1) / 2
                    nxt[j] += half
                    nxt[j + 1] += half
            row = nxt
        return min(1, row[query_glass])

print(Solution().champagneTower(4, 2, 1))

0.5


**Độ phức tạp:** Time `O(query_row^2)`, Space `O(query_row)`.

### 7.4. Cách 5 — DP số nguyên đã nhân `2^i` (chính xác tuyệt đối)

**Ý tưởng.** Ở hàng `i` mọi lượng vào đều có dạng `k / 2^i` với `k` nguyên. Nếu **nhân mọi giá trị ở hàng `i` với `2^i`** thì ta chỉ làm việc với **số nguyên**. Gọi `B[i][j] = inflow(i, j) * 2^i`. Ly đầy khi `B > 2^i`; phần tràn thực là `x - 1`, mỗi con nhận `(x - 1) / 2`, nhân `2^(i+1)` ra **`B - 2^i`** (số nguyên, không có chia!). Vậy:

- `B[0][0] = poured`.
- Nếu `B[i][j] > 2^i` thì `B[i+1][j] += B[i][j] - 2^i` và `B[i+1][j+1] += B[i][j] - 2^i`.
- Đáp án chính xác: `min(1, Fraction(B[q][g], 2^q))`.

Không có phép chia, không có sai số làm tròn. Ta dùng nó làm **oracle** để kiểm tra các cách dùng số thực.

In [18]:
def champagne_exact(poured, query_row, query_glass):
    row = [poured]                              # row[j] = inflow * 2^i (số nguyên)
    for i in range(query_row):
        cap = 1 << i                            # 2^i: sức chứa 1 cốc sau khi nhân 2^i
        nxt = [0] * (len(row) + 1)
        for j, x in enumerate(row):
            if x > cap:
                extra = x - cap                 # = (inflow - 1) * 2^i, và mỗi con nhận đúng extra (ở thang 2^(i+1))
                nxt[j] += extra
                nxt[j + 1] += extra
        row = nxt
    return min(Fraction(1), Fraction(row[query_glass], 1 << query_row))

for args, expected in examples:
    got = champagne_exact(*args)
    print(args, "->", got, "=", float(got))
    assert abs(float(got) - expected) < 1e-12
print(champagne_exact(4, 2, 1), champagne_exact(4, 2, 0), champagne_exact(25, 6, 1))

(1, 1, 1) -> 0 = 0.0
(2, 1, 1) -> 1/2 = 0.5
(100000009, 33, 17) -> 1 = 1.0
1/2 1/4 3/16


In [19]:
class Solution:
    def champagneTower(self, poured: int, query_row: int, query_glass: int) -> float:
        row = [poured]
        for i in range(query_row):
            cap = 1 << i
            nxt = [0] * (len(row) + 1)
            for j, x in enumerate(row):
                if x > cap:
                    nxt[j] += x - cap
                    nxt[j + 1] += x - cap
            row = nxt
        return min(1.0, row[query_glass] / (1 << query_row))

print(Solution().champagneTower(25, 6, 1))

0.1875


**Độ phức tạp:** Time `O(query_row^2)` phép số nguyên (Python có số nguyên lớn tùy ý nên không tràn; số lớn nhất khoảng `10^9 * 2^99`, tức ~130 bit — vẫn nhẹ), Space `O(query_row)`. Ưu điểm: **chính xác**; nhược điểm: số nguyên lớn hơn.

## 8. TRACE từng bước

### 8.1. Trace DP hai chiều với `poured = 4`, `query_row = 2`, `query_glass = 1`

Ban đầu `f[0][0] = 4`. Bảng `f` (chỉ các ô liên quan), `half = (f - 1) / 2`:

| `(i, j)` | `f[i][j]` trước | `> 1`? | `half` | ghi vào | `f[i][j]` sau |
|---|---|---|---|---|---|
| (0,0) | 4 | có | 1.5 | `f[1][0] += 1.5`, `f[1][1] += 1.5` | 1 |
| (1,0) | 1.5 | có | 0.25 | `f[2][0] += 0.25`, `f[2][1] += 0.25` | 1 |
| (1,1) | 1.5 | có | 0.25 | `f[2][1] += 0.25`, `f[2][2] += 0.25` | 1 |
| (2,0) | 0.25 | không | | | 0.25 |
| (2,1) | 0.5 | không | | | 0.5 |
| (2,2) | 0.25 | không | | | 0.25 |

Kết quả `f[2][1] = 0.5`.

In [20]:
def trace_dp(poured, query_row, query_glass):
    f = [[0] * (query_row + 2) for _ in range(query_row + 2)]
    f[0][0] = poured
    for i in range(query_row + 1):
        for j in range(i + 1):
            before = f[i][j]
            if before > 1:
                half = (before - 1) / 2
                f[i][j] = 1
                f[i + 1][j] += half
                f[i + 1][j + 1] += half
                print(f"({i},{j}): {before} > 1 -> half = {half}; f[{i + 1}][{j}] += {half}, f[{i + 1}][{j + 1}] += {half}")
            else:
                print(f"({i},{j}): {before} <= 1 -> giữ nguyên, không tràn")
    print("hàng cuối:", f[query_row][:query_row + 1])
    return f[query_row][query_glass]

assert trace_dp(4, 2, 1) == 0.5

(0,0): 4 > 1 -> half = 1.5; f[1][0] += 1.5, f[1][1] += 1.5
(1,0): 1.5 > 1 -> half = 0.25; f[2][0] += 0.25, f[2][1] += 0.25
(1,1): 1.5 > 1 -> half = 0.25; f[2][1] += 0.25, f[2][2] += 0.25
(2,0): 0.25 <= 1 -> giữ nguyên, không tràn
(2,1): 0.5 <= 1 -> giữ nguyên, không tràn
(2,2): 0.25 <= 1 -> giữ nguyên, không tràn
hàng cuối: [0.25, 0.5, 0.25]


### 8.2. Trace DP số nguyên (nhân `2^i`) với `poured = 4`

| hàng `i` | `row` (đã nhân `2^i`) | thang `2^i` | giá trị thật |
|---|---|---|---|
| 0 | `[4]` | 1 | `[4]` |
| 1 | `[3, 3]` (vì `4 - 1 = 3`) | 2 | `[1.5, 1.5]` |
| 2 | `[1, 2, 1]` (vì `3 - 2 = 1` mỗi cha) | 4 | `[0.25, 0.5, 0.25]` |

Đáp án `(2, 1)`: `2 / 4 = 0.5`.

In [21]:
def trace_exact(poured, query_row):
    row = [poured]
    for i in range(query_row + 1):
        scale = 1 << i
        print(f"hàng {i}: row = {row} (thang {scale}) -> giá trị thật {[Fraction(x, scale) for x in row]}")
        if i == query_row:
            break
        nxt = [0] * (len(row) + 1)
        for j, x in enumerate(row):
            if x > scale:
                nxt[j] += x - scale
                nxt[j + 1] += x - scale
        row = nxt
    return row

assert trace_exact(4, 2) == [1, 2, 1]
print()
trace_exact(25, 4)

hàng 0: row = [4] (thang 1) -> giá trị thật [Fraction(4, 1)]
hàng 1: row = [3, 3] (thang 2) -> giá trị thật [Fraction(3, 2), Fraction(3, 2)]
hàng 2: row = [1, 2, 1] (thang 4) -> giá trị thật [Fraction(1, 4), Fraction(1, 2), Fraction(1, 4)]

hàng 0: row = [25] (thang 1) -> giá trị thật [Fraction(25, 1)]
hàng 1: row = [24, 24] (thang 2) -> giá trị thật [Fraction(12, 1), Fraction(12, 1)]
hàng 2: row = [22, 44, 22] (thang 4) -> giá trị thật [Fraction(11, 2), Fraction(11, 1), Fraction(11, 2)]
hàng 3: row = [18, 58, 58, 18] (thang 8) -> giá trị thật [Fraction(9, 4), Fraction(29, 4), Fraction(29, 4), Fraction(9, 4)]
hàng 4: row = [10, 60, 100, 60, 10] (thang 16) -> giá trị thật [Fraction(5, 8), Fraction(15, 4), Fraction(25, 4), Fraction(15, 4), Fraction(5, 8)]


[10, 60, 100, 60, 10]

### 8.3. Trace "bảng đủ" cho `poured = 25`, `query_row = 6`

`poured = 25` là ca nổi tiếng (đáp án ly `(6, 1)` là `0.1875 = 3/16`). In cả tháp (lượng ly giữ) để thấy hình dạng.

In [22]:
def tower_rows(poured, rows):
    """Trả về list các hàng (lượng mỗi ly giữ, dạng Fraction) cho `rows` hàng, và lượng rơi xuống sàn."""
    cur = [Fraction(poured)]
    result = []
    for i in range(rows):
        kept = [min(Fraction(1), x) for x in cur]
        result.append(kept)
        nxt = [Fraction(0)] * (len(cur) + 1)
        for j, x in enumerate(cur):
            if x > 1:
                nxt[j] += (x - 1) / 2
                nxt[j + 1] += (x - 1) / 2
        cur = nxt
    spill = sum(cur, Fraction(0))
    return result, spill

tower, spill = tower_rows(25, 7)
for i, row in enumerate(tower):
    print(" " * (7 - i) * 3 + "  ".join(f"{str(x):>5}" for x in row))
print("rơi xuống sàn:", spill)
assert tower[6][1] == Fraction(3, 16)

                         1
                      1      1
                   1      1      1
                1      1      1      1
           5/8      1      1      1    5/8
          0      1      1      1      1      0
       0   3/16      1      1      1   3/16      0
rơi xuống sàn: 27/8


## 9. Code nộp LeetCode

Lời giải nộp là **DP hai chiều** (Cách 3): ngắn, chạy tức thì (`~5000` ly). Kết quả là số thực; LeetCode chấp nhận sai số `1e-5`.

In [23]:
class Solution:
    def champagneTower(self, poured: int, query_row: int, query_glass: int) -> float:
        f = [[0] * 101 for _ in range(101)]
        f[0][0] = poured
        for i in range(query_row + 1):
            for j in range(i + 1):
                if f[i][j] > 1:
                    half = (f[i][j] - 1) / 2
                    f[i][j] = 1
                    f[i + 1][j] += half
                    f[i + 1][j + 1] += half
        return f[query_row][query_glass]


sol = Solution()
for args, expected in examples:
    got = sol.champagneTower(*args)
    print(args, "->", got)
    assert abs(got - expected) < 1e-5            # số thực: so sánh với dung sai

(1, 1, 1) -> 0
(2, 1, 1) -> 0.5
(100000009, 33, 17) -> 1


## 10. Vì sao đúng

**Mệnh đề 1 (kết quả không phụ thuộc thứ tự rót).** Xét **toàn bộ** lượng rót `poured` coi như đổ một lần vào ly `(0,0)`. Ly `(0,0)` giữ `min(1, poured)` và đẩy xuống **mỗi con** `max(0, poured - 1) / 2`. Tương tự mỗi ly: nếu tổng lượng chảy vào là `x` thì nó giữ `min(1, x)` và đẩy `max(0, x - 1) / 2` xuống mỗi con. Lập luận bằng quy nạp theo hàng: tổng chảy vào hàng `i + 1` bằng tổng phần tràn của hàng `i`, vì ly chỉ tràn **sau khi đầy** và phần tràn chia đều.

So với mô phỏng từng cốc: mỗi cốc rót vào được chia cho các ly theo cùng quy tắc "đầy mới tràn, tràn chia đều". Tổng chảy vào và tổng giữ lại mỗi ly sau `poured` cốc **bằng** tổng khi đổ một lần (vì phép cộng giao hoán và quy tắc tràn chỉ phụ thuộc ngưỡng tổng). Ta kiểm chứng điều này bằng code: so mô phỏng từng cốc với công thức gộp trên nhiều `(poured, row, glass)`.

**Mệnh đề 2 (DP đúng).** `f[i][j]` (lượng vào) được tính từ `f[i-1][j-1]` và `f[i-1][j]` bằng đúng công thức trên; mỗi ly được xử lý sau khi cả hai cha đã xử lý (duyệt hàng tăng dần). Ly ở hàng `query_row` cũng được cắt về `1` trong vòng lặp nên giá trị trả về luôn `<= 1`.

**Mệnh đề 3 (chính xác của cách số nguyên).** Quy nạp theo hàng: `B[i][j] = inflow(i, j) * 2^i` là số nguyên. Cơ sở: `B[0][0] = poured`. Bước: con nhận `(inflow - 1) / 2` từ mỗi cha đầy; nhân `2^(i+1)`: `(inflow - 1) * 2^i = B - 2^i`, nguyên. Tổng hai cha cho `B` của con (nguyên). Vậy mọi giá trị là số nguyên, không có sai số; chia cho `2^q` cuối cùng ra `Fraction` chính xác.

**Về sai số của số thực.** Mọi lượng ở hàng `i` là bội của `2^-i`; `float` biểu diễn chính xác khi tử số `< 2^53`. Với `poured <= 10^9` và hàng nhỏ, hoàn toàn chính xác; ở hàng sâu có thể có sai số cỡ `10^-16` tương đối — rất nhỏ so với dung sai `10^-5` của LeetCode. Ta đo sai số thực tế ở mục 12.

In [24]:
# Kiểm chứng Mệnh đề 1: mô phỏng từng cốc == công thức gộp == số nguyên chính xác
rng = random.Random(10)
cnt = 0
for _ in range(400):
    poured = rng.randint(0, 200)
    r = rng.randint(0, 8)
    q = rng.randint(0, r)
    a = champagne_cups(poured, r, q)
    b = champagne_dp2d(poured, r, q)
    c = champagne_exact(poured, r, q)
    assert a == b == float(c), (poured, r, q, a, b, c)    # cả ba cùng CHÍNH XÁC (các phân số 2^-k nhỏ)
    cnt += 1
print("đã kiểm", cnt, "bộ (poured, row, glass): mô phỏng từng cốc == DP == số nguyên")

đã kiểm 400 bộ (poured, row, glass): mô phỏng từng cốc == DP == số nguyên


## 11. Time / Space Complexity

Gọi `R = query_row` (`<= 99`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng cốc | `O(poured * R^2)` | `O(R^2)` |
| 2. Đệ quy có nhớ | `O(R^2)` | `O(R^2)` |
| 3. DP 2 chiều (nộp) | `O(R^2)` | `O(R^2)` (kích thước cố định `101 x 101`) |
| 4. DP cuộn hàng | `O(R^2)` | `O(R)` |
| 5. DP số nguyên nhân `2^i` | `O(R^2)` | `O(R)` (số nguyên lớn ~130 bit) |

Trường hợp lớn nhất: `R = 99` chỉ khoảng `5000` ly. Ta đếm số ô được xử lý.

In [25]:
def count_cells(query_row):
    return sum(i + 1 for i in range(query_row + 1))

for r in [0, 1, 9, 49, 99]:
    print(f"query_row = {r}: số ly xử lý = {count_cells(r)} (= (r+1)(r+2)/2 = {(r + 1) * (r + 2) // 2})")
print("cỡ số nguyên lớn nhất ở cách 5:", (10 ** 9 * 2 ** 99).bit_length(), "bit")

query_row = 0: số ly xử lý = 1 (= (r+1)(r+2)/2 = 1)
query_row = 1: số ly xử lý = 3 (= (r+1)(r+2)/2 = 3)
query_row = 9: số ly xử lý = 55 (= (r+1)(r+2)/2 = 55)
query_row = 49: số ly xử lý = 1275 (= (r+1)(r+2)/2 = 1275)
query_row = 99: số ly xử lý = 5050 (= (r+1)(r+2)/2 = 5050)
cỡ số nguyên lớn nhất ở cách 5: 129 bit


## 12. Bộ test

Vì kết quả là **số thực**, mọi so sánh dùng dung sai `1e-9` (hoặc dùng `Fraction` để so **chính xác**). Gồm: ví dụ đề, biên (`poured = 0`, ly trên cùng, ly rìa trái/phải, hàng sâu nhất), phản ví dụ cho lời giải sai, đối chiếu 5 cách, các **tính chất** (bảo toàn lượng, đối xứng trái-phải, đơn điệu theo `poured`) kiểm bằng `Fraction`, test ngẫu nhiên có seed đúng ràng buộc, và test các giá trị cực đại.

In [26]:
TOL = 1e-9
ALL_METHODS = {
    "cups": champagne_cups,         # chỉ dùng khi poured nhỏ
    "memo": champagne_memo,
    "dp2d": champagne_dp2d,
    "rolling": champagne_rolling,
    "exact": lambda p, r, q: float(champagne_exact(p, r, q)),
    "leetcode": Solution().champagneTower,
}
FAST = [k for k in ALL_METHODS if k != "cups"]

fixed_cases = [
    ((1, 1, 1), 0.0),
    ((2, 1, 1), 0.5),
    ((100000009, 33, 17), 1.0),
    ((0, 0, 0), 0.0),               # không rót gì
    ((1, 0, 0), 1.0),               # đúng một cốc: ly trên cùng đầy
    ((2, 0, 0), 1.0),               # ly trên cùng không vượt quá 1
    ((2, 1, 0), 0.5),
    ((3, 1, 0), 1.0),
    ((3, 2, 1), 0.0),
    ((4, 2, 0), 0.25),
    ((4, 2, 1), 0.5),
    ((4, 2, 2), 0.25),
    ((25, 6, 1), 0.1875),           # ca nổi tiếng
    ((8, 3, 0), 0.125),
    ((8, 3, 1), 0.875),
    ((10 ** 9, 99, 0), None),       # biên lớn nhất, kiểm bằng so sánh chéo
    ((10 ** 9, 99, 99), None),
    ((10 ** 9, 99, 49), None),
]
for (p, r, q), expected in fixed_cases:
    exact = float(champagne_exact(p, r, q))
    if expected is not None:
        assert abs(exact - expected) < TOL, ((p, r, q), exact, expected)
    for name in FAST:
        got = ALL_METHODS[name](p, r, q)
        assert abs(got - exact) < TOL, (name, (p, r, q), got, exact)
    if p <= 300 and r <= 9:
        assert abs(ALL_METHODS["cups"](p, r, q) - exact) < TOL
print("test cố định:", len(fixed_cases), "bộ: OK")
print("giá trị cực đại:", [champagne_dp2d(10 ** 9, 99, q) for q in (0, 49, 99)])

test cố định: 18 bộ: OK
giá trị cực đại: [0, 1, 0]


In [27]:
# Phản ví dụ 1: quên TRỪ 1 (chia đôi cả lượng vào)
def wrong_no_minus_one(poured, query_row, query_glass):
    f = [[0] * 101 for _ in range(101)]
    f[0][0] = poured
    for i in range(query_row + 1):
        for j in range(i + 1):
            if f[i][j] > 1:
                half = f[i][j] / 2               # SAI: phải là (f - 1) / 2
                f[i][j] = 1
                f[i + 1][j] += half
                f[i + 1][j + 1] += half
    return f[query_row][query_glass]

print("quên trừ 1:", wrong_no_minus_one(4, 2, 1), "| đúng:", champagne_dp2d(4, 2, 1))
print("quên trừ 1:", wrong_no_minus_one(2, 1, 1), "| đúng:", champagne_dp2d(2, 1, 1))

# Phản ví dụ 2: chia nguyên //
def wrong_floor_div(poured, query_row, query_glass):
    f = [[0] * 101 for _ in range(101)]
    f[0][0] = poured
    for i in range(query_row + 1):
        for j in range(i + 1):
            if f[i][j] > 1:
                half = (f[i][j] - 1) // 2        # SAI: mất phần nửa
                f[i][j] = 1
                f[i + 1][j] += half
                f[i + 1][j + 1] += half
    return f[query_row][query_glass]

print("dùng //:", wrong_floor_div(2, 1, 1), "| đúng:", champagne_dp2d(2, 1, 1))

# Phản ví dụ 3: quên cắt ở 1 (chỉ lặp tới hàng query_row - 1)
def wrong_no_cap(poured, query_row, query_glass):
    f = [[0] * 101 for _ in range(101)]
    f[0][0] = poured
    for i in range(query_row):                   # SAI: bỏ hàng query_row nên ly truy vấn không bị cắt
        for j in range(i + 1):
            if f[i][j] > 1:
                half = (f[i][j] - 1) / 2
                f[i][j] = 1
                f[i + 1][j] += half
                f[i + 1][j + 1] += half
    return f[query_row][query_glass]

print("quên cắt ở 1:", wrong_no_cap(100000009, 33, 17), "| đúng:", champagne_dp2d(100000009, 33, 17))

quên trừ 1: 1 | đúng: 0.5
quên trừ 1: 1.0 | đúng: 0.5
dùng //: 0 | đúng: 0.5
quên cắt ở 1: 13583347.932617264 | đúng: 1


In [28]:
# Các TÍNH CHẤT, kiểm CHÍNH XÁC bằng Fraction
rng = random.Random(799)

def tower_exact_rows(poured, rows):
    """Trả về (các hàng lượng ly giữ, lượng rơi xuống sàn) bằng Fraction."""
    return tower_rows(poured, rows)

for _ in range(200):
    poured = rng.randint(0, 5000)
    R = rng.randint(1, 30)
    rows_kept, spill = tower_exact_rows(poured, R)
    # (1) Bảo toàn: tổng lượng các ly giữ + lượng rơi xuống sàn == poured
    total = sum((sum(r, Fraction(0)) for r in rows_kept), Fraction(0)) + spill
    assert total == poured, (poured, R, total)
    # (2) Đối xứng trái-phải
    for i, row in enumerate(rows_kept):
        assert row == row[::-1], (poured, i)
    # (3) Ly nằm trong khoảng [0, 1]
    assert all(0 <= x <= 1 for row in rows_kept for x in row)
    # (4) Đơn điệu theo poured: rót thêm không làm ly nào ít đi
    more, _ = tower_exact_rows(poured + rng.randint(1, 50), R)
    assert all(a <= b for ra, rb in zip(rows_kept, more) for a, b in zip(ra, rb))
    # (5) Ly nào có giá trị dương ở hàng i thì ly ở hàng cha (trên) phải ĐẦY
    for i in range(1, R):
        for j, x in enumerate(rows_kept[i]):
            if x > 0:
                parents = [rows_kept[i - 1][k] for k in (j - 1, j) if 0 <= k <= i - 1]
                assert any(p == 1 for p in parents), (poured, i, j)
print("bảo toàn / đối xứng / [0,1] / đơn điệu / cha-đầy-trước: đúng trên 200 tháp ngẫu nhiên")

bảo toàn / đối xứng / [0,1] / đơn điệu / cha-đầy-trước: đúng trên 200 tháp ngẫu nhiên


In [29]:
# Đối chiếu 5 cách trên rất nhiều bộ (poured, row, glass) ngẫu nhiên đúng ràng buộc
rng = random.Random(7990)
max_err = 0.0
for _ in range(3000):
    poured = rng.choice([rng.randint(0, 60), rng.randint(0, 10 ** 5), rng.randint(0, 10 ** 9), 10 ** 9, 2 ** rng.randint(0, 29)])
    r = rng.choice([rng.randint(0, 5), rng.randint(0, 99), 99])
    q = rng.randint(0, r)
    exact = float(champagne_exact(poured, r, q))
    for name in FAST:
        got = ALL_METHODS[name](poured, r, q)
        err = abs(got - exact)
        max_err = max(max_err, err)
        assert err < TOL, (name, (poured, r, q), got, exact)
    if poured <= 300 and r <= 9:
        assert abs(champagne_cups(poured, r, q) - exact) < TOL
print("3000 bộ ngẫu nhiên: mọi cách khớp oracle chính xác; sai số lớn nhất của số thực =", max_err)

# Vét cạn: mọi poured trong [0, 70], mọi ly của 8 hàng đầu
cnt = 0
for p in range(0, 71):
    for r in range(0, 8):
        for q in range(r + 1):
            ex = float(champagne_exact(p, r, q))
            assert abs(champagne_cups(p, r, q) - ex) < 1e-12
            assert abs(champagne_memo(p, r, q) - ex) < 1e-12
            assert abs(champagne_rolling(p, r, q) - ex) < 1e-12
            assert abs(Solution().champagneTower(p, r, q) - ex) < 1e-12
            cnt += 1
print("vét cạn", cnt, "bộ nhỏ: các cách khớp tới 1e-12")

3000 bộ ngẫu nhiên: mọi cách khớp oracle chính xác; sai số lớn nhất của số thực = 4.0939474033052647e-16


vét cạn 2556 bộ nhỏ: các cách khớp tới 1e-12


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ 1 trước khi chia đôi

Ly đầy **giữ lại 1 cốc**; chỉ phần **vượt quá 1** mới tràn. Công thức đúng `(x - 1) / 2`, không phải `x / 2`. Ví dụ `poured = 2`: ly trên cùng giữ `1`, tràn `1`, mỗi con nhận `0.5`. Nếu quên trừ, mỗi con nhận `1` — sai (đã thấy ở mục 12: `wrong_no_minus_one`).

### Lỗi 2 — Quên cắt kết quả ở `1`

Ly chỉ chứa tối đa `1` cốc nhưng lượng **chảy vào** có thể rất lớn (`poured = 10^9`). Nếu trả `f[query_row][query_glass]` mà **chưa cắt**, có thể ra số lớn hơn `1` (như `wrong_no_cap` ở mục 12). Dùng `min(1, ...)` hoặc xử lý cả hàng `query_row` trong vòng lặp (gán `f[i][j] = 1`).

### Lỗi 3 — Mảng quá nhỏ gây `IndexError`

Khi `query_row = 99`, ta xử lý `i = 99` và ghi vào `f[100][...]`. Mảng `100 x 100` là **không đủ**; cần `101 x 101` (hoặc chỉ ghi xuống hàng dưới khi `i < query_row`).

In [30]:
def wrong_small_array(poured, query_row, query_glass):
    f = [[0] * 100 for _ in range(100)]          # SAI: thiếu một hàng/cột
    f[0][0] = poured
    for i in range(query_row + 1):
        for j in range(i + 1):
            if f[i][j] > 1:
                half = (f[i][j] - 1) / 2
                f[i][j] = 1
                f[i + 1][j] += half               # i = 99 -> f[100] không tồn tại
                f[i + 1][j + 1] += half
    return f[query_row][query_glass]

print(wrong_small_array(10, 50, 3))
try:
    print(wrong_small_array(10 ** 9, 99, 5))
except IndexError as e:
    print("IndexError:", e)

0
IndexError: list index out of range


### Lỗi 4 — Dùng chia nguyên `//` thay vì `/`

`(x - 1) // 2` làm mất phần `0.5`, `0.25`, ... (đã thấy ở mục 12: ví dụ 2 cho `0` thay vì `0.5`).

### Lỗi 5 — Đẩy lượng **âm** xuống ly dưới

Nếu không kiểm tra `x > 1` mà luôn cộng `(x - 1) / 2` cho con, thì với ly **chưa đầy** (`x < 1`) ta cộng một số **âm**, làm ly con "mất" champagne.

In [31]:
def wrong_negative(poured, query_row, query_glass):
    f = [[0] * 101 for _ in range(101)]
    f[0][0] = poured
    for i in range(query_row + 1):
        for j in range(i + 1):
            half = (f[i][j] - 1) / 2             # SAI: không kiểm tra f[i][j] > 1
            f[i][j] = min(1, f[i][j])
            f[i + 1][j] += half
            f[i + 1][j + 1] += half
    return f[query_row][query_glass]

for args in [(4, 2, 1), (1, 1, 1), (2, 2, 1)]:
    print(args, "| sai:", wrong_negative(*args), "| đúng:", champagne_dp2d(*args))

(4, 2, 1) | sai: 0.5 | đúng: 0.5
(1, 1, 1) | sai: 0.0 | đúng: 0
(2, 2, 1) | sai: -0.5 | đúng: 0


### Lỗi 6 — So sánh số thực bằng `==`

Khi viết test, `got == 0.1875` có thể đúng hoặc sai tùy cách tính (với phân số có mẫu `2^k` thường chính xác, nhưng đừng dựa vào). Luôn dùng `abs(got - expected) < 1e-9` (hoặc `math.isclose`) hoặc so sánh bằng `Fraction`.

In [32]:
import math
x = 0.1 + 0.2
print(x == 0.3, math.isclose(x, 0.3), abs(x - 0.3) < 1e-9)
print(Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))

False True True
True


## 14. Mẹo & nhận dạng pattern

- **"Chất lỏng/lượng chảy xuống các tầng, chia đều"** → **DP theo hàng**: mỗi ô nhận từ hai ô cha ở hàng trên (giống tam giác Pascal).
- Khi đầu vào lớn (`10^9`) nhưng cấu trúc nhỏ (`~5000` ly): đừng mô phỏng **từng đơn vị**; hãy tính **tổng lượng** đi qua mỗi nút.
- **Tam giác Pascal** liên quan: nếu mọi ly đều tràn thì lượng vào ly `(i, j)` tỉ lệ với `C(i, j) / 2^i`.
- Kiểm tra số thực: dùng **oracle chính xác** (`Fraction` hoặc số nguyên đã nhân tỉ lệ) và so với dung sai.
- Mẹo số nguyên: nếu mọi phép chia đều là chia đôi, **nhân thang `2^i`** để tránh chia và tránh sai số.
- Họ bài liên quan: *Pascal's Triangle* (118, 119), *Minimum Path Sum* (64), *Triangle* (120), *Unique Paths* (62).

## 15. Biến thể

### 15.1. Đếm số ly **đầy** trong `R` hàng đầu

Một ly là đầy khi lượng vào `>= 1`.

### 15.2. In **toàn bộ một hàng**

Trả về lượng của mọi ly ở hàng `i` (bằng DP cuộn hàng).

### 15.3. Lượng champagne **rơi xuống sàn** sau hàng `R - 1`

Tổng phần tràn của hàng cuối. Cùng với tổng ly giữ, bảo toàn: giữ + rơi = `poured`.

### 15.4. Số cốc **tối thiểu** để ly `(r, g)` đầy (Bài 5)

Hàm "đầy" đơn điệu theo `poured` nên tìm bằng tìm kiếm nhị phân (hoặc tăng gấp đôi rồi nhị phân).

In [33]:
def full_glasses(poured, rows):
    """15.1: số ly đầy (>= 1 cốc lượng vào) trong `rows` hàng đầu."""
    tower, _ = tower_rows(poured, rows)
    return sum(1 for row in tower for x in row if x == 1)

def row_amounts(poured, i):
    """15.2: lượng ly giữ ở hàng i (list float)."""
    tower, _ = tower_rows(poured, i + 1)
    return [float(x) for x in tower[i]]

def spill_to_floor(poured, rows):
    """15.3: lượng rơi xuống sàn sau `rows` hàng (Fraction)."""
    return tower_rows(poured, rows)[1]

print(full_glasses(4, 3), full_glasses(7, 3), full_glasses(10 ** 9, 5))
print(row_amounts(25, 6))
print(spill_to_floor(4, 2), spill_to_floor(25, 6))

rng = random.Random(15)
for _ in range(300):
    p, R = rng.randint(0, 3000), rng.randint(1, 20)
    tower, spill = tower_rows(p, R)
    kept = sum((sum(r, Fraction(0)) for r in tower), Fraction(0))
    assert kept + spill_to_floor(p, R) == p
    assert full_glasses(p, R) == sum(1 for r in range(R) for q in range(r + 1) if champagne_exact(p, r, q) == 1)
    assert row_amounts(p, R - 1) == [champagne_dp2d(p, R - 1, q) for q in range(R)]
print("các biến thể 15.1-15.3 OK")

3 6 15
[0.0, 0.1875, 1.0, 1.0, 1.0, 0.1875, 0.0]
1 27/4


các biến thể 15.1-15.3 OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng cốc | Rót từng cốc, tràn đệ quy | `O(poured R^2)` | `O(R^2)` | Chỉ để hiểu đề (`poured` nhỏ) |
| 2. Đệ quy có nhớ | `inflow(i, j)` theo công thức, memo | `O(R^2)` | `O(R^2)` | Viết thẳng từ công thức |
| 3. DP 2 chiều | Duyệt hàng, đẩy tràn xuống con | `O(R^2)` | `O(R^2)` | **Nộp bài**, quen thuộc |
| 4. DP cuộn hàng | Chỉ giữ một hàng | `O(R^2)` | `O(R)` | Tiết kiệm bộ nhớ |
| 5. DP số nguyên `2^i` | Nhân thang để tránh chia | `O(R^2)` | `O(R)` | Cần kết quả **chính xác tuyệt đối** / oracle |

In [34]:
rng = random.Random(16)
cases = [(rng.randint(0, 10 ** 9), rng.randint(0, 99)) for _ in range(5)]
for p, r in cases:
    q = rng.randint(0, r)
    vals = {name: ALL_METHODS[name](p, r, q) for name in FAST}
    print((p, r, q), {k: round(v, 12) for k, v in vals.items()})
    assert max(vals.values()) - min(vals.values()) < TOL

(388182122, 60, 45) {'memo': 1, 'dp2d': 1, 'rolling': 1, 'exact': 1.0, 'leetcode': 1}
(515912189, 36, 16) {'memo': 1, 'dp2d': 1, 'rolling': 1, 'exact': 1.0, 'leetcode': 1}
(447699478, 29, 7) {'memo': 1, 'dp2d': 1, 'rolling': 1, 'exact': 1.0, 'leetcode': 1}
(479704103, 0, 0) {'memo': 1, 'dp2d': 1, 'rolling': 1, 'exact': 1.0, 'leetcode': 1}
(439714165, 84, 1) {'memo': 0, 'dp2d': 0, 'rolling': 0, 'exact': 0.0, 'leetcode': 0}


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(Solution().champagneTower(3, 1, 0), Solution().champagneTower(3, 2, 1), Solution().champagneTower(0, 0, 0))
print(0.1 + 0.2 == 0.3, 1 / 2 + 1 / 4 == 0.75)
print(5 / 2, 5 // 2, max(0, 0.5 - 1) / 2)
```

In [35]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `poured = 8`, hãy tính tay lượng ở **mọi ly của hàng 3** (4 ly). Gợi ý: tính lần lượt hàng 0, 1, 2, rồi hàng 3.

In [36]:
# Bài 2: điền kết quả tính tay (list 4 số thực)
hand_row3 = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DP hai chiều.

In [37]:
def exercise_3_champagne(poured: int, query_row: int, query_glass: int) -> float:
    f = [[0] * 101 for _ in range(101)]
    f[0][0] = poured
    # TODO 1: lặp i từ 0 đến query_row (bao gồm), j từ 0 đến i
    #     TODO 2: nếu f[i][j] > 1:
    #         TODO 3: half = (f[i][j] - 1) / 2, rồi đặt f[i][j] = 1
    #         TODO 4: cộng half vào f[i + 1][j] và f[i + 1][j + 1]
    return f[query_row][query_glass]


# assert abs(exercise_3_champagne(2, 1, 1) - 0.5) < 1e-9
# assert abs(exercise_3_champagne(4, 2, 1) - 0.5) < 1e-9
# assert abs(exercise_3_champagne(25, 6, 1) - 0.1875) < 1e-9

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 4, hãy tự viết `exercise_4_rolling(poured, query_row, query_glass)` dùng **một hàng** (`O(query_row)` bộ nhớ).

In [38]:
def exercise_4_rolling(poured: int, query_row: int, query_glass: int) -> float:
    # TODO
    pass


# assert abs(exercise_4_rolling(2, 1, 1) - 0.5) < 1e-9
# assert abs(exercise_4_rolling(100000009, 33, 17) - 1.0) < 1e-9
# assert abs(exercise_4_rolling(8, 3, 1) - 0.875) < 1e-9

### Bài 5 — Mini challenge: số cốc **tối thiểu** để ly `(r, g)` đầy (khó)

Viết `exercise_5_min_poured(r, g)` trả về **số nguyên `poured` nhỏ nhất** sao cho ly `(r, g)` đầy (lượng vào `>= 1`). Gợi ý: hàm "ly đầy" **đơn điệu** theo `poured`; dùng DP chính xác (số nguyên nhân `2^i`) làm hàm kiểm tra, tăng `hi` gấp đôi tới khi đầy, rồi tìm kiếm nhị phân.

```python
exercise_5_min_poured(0, 0) -> 1
exercise_5_min_poured(1, 0) -> 3
exercise_5_min_poured(2, 1) -> 5
exercise_5_min_poured(2, 0) -> 7
```

In [39]:
def exercise_5_min_poured(r: int, g: int) -> int:
    # TODO
    pass


# assert exercise_5_min_poured(0, 0) == 1
# assert exercise_5_min_poured(1, 0) == 3
# assert exercise_5_min_poured(2, 1) == 5
# assert exercise_5_min_poured(2, 0) == 7

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `poured = 3` làm hai ly hàng 1 vừa đầy, không còn tràn.
- Mức 2: `0.1 + 0.2` không bằng `0.3`; còn `1/2 + 1/4` là bội của `2^-2` nên chính xác. `max(0, -0.5) = 0`.
- Mức 3: dòng 1 `1 0 0`; dòng 2 `False True`; dòng 3 `2.5 2 0.0`.

**Bài 2**
- Mức 1: hàng 0 nhận `8`, tràn `7`, mỗi con `3.5`.
- Mức 2: hàng 1 mỗi ly `3.5` tràn `2.5` → mỗi con nhận `1.25`; ly giữa hàng 2 nhận `1.25 + 1.25 = 2.5`; ly ngoài `1.25`. Hàng 2 tràn: ngoài `0.25` (con nhận `0.125`), giữa `1.5` (con nhận `0.75`).
- Mức 3: hàng 3 = `[0.125, 0.125 + 0.75, 0.75 + 0.125, 0.125] = [0.125, 0.875, 0.875, 0.125]`.

**Bài 3**
- Mức 1: giống Cách 3 (mục 7.2).
- Mức 2: `for i in range(query_row + 1): for j in range(i + 1):`.
- Mức 3: `if f[i][j] > 1: half = (f[i][j] - 1) / 2; f[i][j] = 1; f[i + 1][j] += half; f[i + 1][j + 1] += half`.

**Bài 4**
- Mức 1: `row = [poured]`; mỗi vòng tạo `nxt` dài hơn 1.
- Mức 2: với mỗi `(j, x)` có `x > 1`: `half = (x - 1) / 2`, cộng vào `nxt[j]` và `nxt[j + 1]`.
- Mức 3: lặp `query_row` lần rồi `return min(1, row[query_glass])`.

**Bài 5**
- Mức 1: viết hàm `is_full(p)` bằng `champagne_exact(p, r, g) == 1`.
- Mức 2: `hi = 1; while not is_full(hi): hi *= 2`; rồi nhị phân trên `[hi // 2, hi]` (hoặc `[1, hi]`).
- Mức 3: `lo, hi = 1, hi`; `while lo < hi: mid = (lo + hi) // 2; if is_full(mid): hi = mid else: lo = mid + 1`; trả `lo`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [40]:
print(Solution().champagneTower(3, 1, 0), Solution().champagneTower(3, 2, 1), Solution().champagneTower(0, 0, 0))
print(0.1 + 0.2 == 0.3, 1 / 2 + 1 / 4 == 0.75)
print(5 / 2, 5 // 2, max(0, 0.5 - 1) / 2)

assert (Solution().champagneTower(3, 1, 0), Solution().champagneTower(3, 2, 1), Solution().champagneTower(0, 0, 0)) == (1, 0, 0)
assert (0.1 + 0.2 == 0.3, 1 / 2 + 1 / 4 == 0.75) == (False, True)
assert (5 / 2, 5 // 2, max(0, 0.5 - 1) / 2) == (2.5, 2, 0.0)

1.0 0 0
False True
2.5 2 0.0


### Đáp án Bài 2

In [41]:
row3 = [Solution().champagneTower(8, 3, q) for q in range(4)]
print(row3)
assert row3 == [0.125, 0.875, 0.875, 0.125]

[0.125, 0.875, 0.875, 0.125]


### Đáp án Bài 3

In [42]:
def exercise_3_solution(poured: int, query_row: int, query_glass: int) -> float:
    f = [[0] * 101 for _ in range(101)]
    f[0][0] = poured
    for i in range(query_row + 1):
        for j in range(i + 1):
            if f[i][j] > 1:
                half = (f[i][j] - 1) / 2
                f[i][j] = 1
                f[i + 1][j] += half
                f[i + 1][j + 1] += half
    return f[query_row][query_glass]

assert abs(exercise_3_solution(2, 1, 1) - 0.5) < 1e-9
assert abs(exercise_3_solution(4, 2, 1) - 0.5) < 1e-9
assert abs(exercise_3_solution(25, 6, 1) - 0.1875) < 1e-9
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_solution(poured: int, query_row: int, query_glass: int) -> float:
    row = [poured]
    for _ in range(query_row):
        nxt = [0] * (len(row) + 1)
        for j, x in enumerate(row):
            if x > 1:
                half = (x - 1) / 2
                nxt[j] += half
                nxt[j + 1] += half
        row = nxt
    return min(1, row[query_glass])

assert abs(exercise_4_solution(2, 1, 1) - 0.5) < 1e-9
assert abs(exercise_4_solution(100000009, 33, 17) - 1.0) < 1e-9
assert abs(exercise_4_solution(8, 3, 1) - 0.875) < 1e-9
rng = random.Random(44)
for _ in range(500):
    p, r = rng.randint(0, 10 ** 9), rng.randint(0, 99)
    q = rng.randint(0, r)
    assert abs(exercise_4_solution(p, r, q) - float(champagne_exact(p, r, q))) < 1e-9
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_solution(r: int, g: int) -> int:
    def is_full(p):
        return champagne_exact(p, r, g) == 1

    hi = 1
    while not is_full(hi):
        hi *= 2
    lo = 1
    while lo < hi:
        mid = (lo + hi) // 2
        if is_full(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

assert exercise_5_solution(0, 0) == 1
assert exercise_5_solution(1, 0) == 3
assert exercise_5_solution(2, 1) == 5
assert exercise_5_solution(2, 0) == 7

# oracle độc lập: tìm tuyến tính từ 1 bằng mô phỏng từng cốc (chỉ cho hàng nhỏ)
def linear_search(r, g):
    p = 1
    while champagne_cups(p, r, g) < 1:
        p += 1
    return p

for r in range(0, 5):
    for g in range(r + 1):
        assert exercise_5_solution(r, g) == linear_search(r, g), (r, g)
print("Bài 5 OK:", {(r, g): exercise_5_solution(r, g) for r in range(4) for g in range(r + 1)})

Bài 5 OK: {(0, 0): 1, (1, 0): 3, (1, 1): 3, (2, 0): 7, (2, 1): 5, (2, 2): 7, (3, 0): 15, (3, 1): 9, (3, 2): 9, (3, 3): 15}


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template DP hai chiều

```python
class Solution:
    def champagneTower(self, poured: int, query_row: int, query_glass: int) -> float:
        f = [[0] * 101 for _ in range(101)]       # 101: vì ghi xuống hàng i + 1
        f[0][0] = poured                          # f[i][j] = lượng CHẢY VÀO ly (i, j)
        for i in range(query_row + 1):
            for j in range(i + 1):
                if f[i][j] > 1:                   # chỉ tràn khi vượt 1
                    half = (f[i][j] - 1) / 2      # TRỪ 1 rồi chia đôi
                    f[i][j] = 1                   # ly chỉ giữ 1
                    f[i + 1][j] += half
                    f[i + 1][j + 1] += half
        return f[query_row][query_glass]
```

Câu thần chú: **"ly đầy giữ 1, phần dư trừ 1 rồi chia đôi cho hai ly con; mỗi ly tính một lần theo hàng."**

### Checklist trước khi submit

- [ ] Trừ 1 trước khi chia đôi: `(x - 1) / 2`; dùng chia thực `/`, không dùng `//`.
- [ ] Chỉ đẩy xuống khi `x > 1` (không đẩy số âm).
- [ ] Ly truy vấn bị cắt về tối đa `1` (xử lý cả hàng `query_row`, hoặc `min(1, ...)`).
- [ ] Mảng đủ lớn: `101 x 101` (hoặc chỉ ghi xuống hàng dưới khi chưa phải hàng cuối).
- [ ] Không mô phỏng từng cốc (`poured` tới `10^9`).
- [ ] Khi kiểm thử số thực: so sánh với dung sai hoặc bằng `Fraction`; test `poured = 0`, `query_row = 0`, ly rìa, `poured = 10^9` ở hàng `99`.
- [ ] Skeleton đúng: `class Solution`, method `champagneTower(self, poured: int, query_row: int, query_glass: int) -> float`.